# Predicting the 2026 FIFA World Cup

An updated take on the original `worldcup_pred_2022.ipynb` notebook for the **2026 FIFA World Cup** (USA / Canada / Mexico).

### What's new in this notebook

1. **Up‑to‑date data** — the original `international_matches.csv` only runs to June 2022, so this notebook layers in `results_recent.csv` (the actively‑maintained `martj42/international_results` dataset) which contains every senior international match through **June 2026**, including matches already played at the current World Cup.
2. **Better ML model** — XGBoost (multi:softprob) replaces the original RandomForest, plus richer features: rolling 5‑match form, average goal differential, running ELO rating (K=24), and rank / FIFA‑points / ELO deltas.
3. **Microsoft Foundry LLM (optional)** — plug any model from the [Azure AI Foundry catalog](https://ai.azure.com/explore/models) into the pipeline for match commentary. Default `gpt-4o`; swap for `gpt-4o-mini`, `o3-mini`, `Phi-4`, `Llama-3.3-70B-Instruct`, or `Mistral-Large-2411`. If env vars aren't set the notebook silently degrades to pure‑ML output.
4. **This week's fixtures** — at the end of the notebook we predict every scheduled 2026 World Cup match in the next 7 days.

## 1. Install / import dependencies

In [1]:
%pip install --quiet pandas numpy scikit-learn xgboost matplotlib seaborn azure-ai-inference azure-identity

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, random, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, log_loss, classification_report
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

Matplotlib is building the font cache; this may take a moment.


## 2. Configure the (optional) Microsoft Foundry model

Pick any model from the [Foundry catalog](https://ai.azure.com/explore/models) — `gpt-4o` is a strong default. Enable LLM features by setting two env vars before launching the notebook:

```bash
export AZURE_AI_ENDPOINT="https://<your-foundry-project>.services.ai.azure.com/models"
export AZURE_AI_API_KEY="<your-key>"
```

If they are missing the notebook silently degrades to ML‑only predictions.

In [3]:
FOUNDRY_MODEL = "gpt-4o"  # try gpt-4o-mini, o3-mini, Phi-4, Llama-3.3-70B-Instruct, Mistral-Large-2411
AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_KEY  = os.getenv("AZURE_AI_API_KEY")

foundry_client = None
try:
    from azure.ai.inference import ChatCompletionsClient
    from azure.ai.inference.models import SystemMessage, UserMessage
    from azure.core.credentials import AzureKeyCredential
    if AZURE_AI_ENDPOINT and AZURE_AI_API_KEY:
        foundry_client = ChatCompletionsClient(endpoint=AZURE_AI_ENDPOINT, credential=AzureKeyCredential(AZURE_AI_API_KEY))
        print(f"\u2705 Foundry client ready using model '{FOUNDRY_MODEL}'")
    else:
        print("\u26a0\ufe0f  AZURE_AI_* env vars not set \u2014 running in ML-only mode.")
except ImportError:
    print("\u26a0\ufe0f  azure-ai-inference not installed \u2014 ML-only mode.")

def ask_foundry(system: str, user: str, max_tokens: int = 300) -> str | None:
    if foundry_client is None: return None
    try:
        r = foundry_client.complete(model=FOUNDRY_MODEL, temperature=0.3, max_tokens=max_tokens,
                                    messages=[SystemMessage(content=system), UserMessage(content=user)])
        return r.choices[0].message.content.strip()
    except Exception as e:
        print(f"Foundry call failed: {e}"); return None

⚠️  azure-ai-inference not installed — ML-only mode.


## 3. Load both datasets

* **`international_matches.csv`** — the original Kaggle dataset (rich FIFA rank + squad scores) up to **June 2022**.
* **`results_recent.csv`** — `martj42/international_results` snapshot through **27 June 2026**, including every 2026 World Cup fixture played or scheduled.

In [4]:
RICH = pd.read_csv("international_matches.csv", parse_dates=["date"])
NEW  = pd.read_csv("results_recent.csv",       parse_dates=["date"])

NAME_MAP = {
    # New dataset name -> Rich dataset name
    "Korea Republic": "South Korea",
    "IR Iran": "Iran",
    "United States": "USA",
    "Czechia": "Czech Republic",
    "Republic of Ireland": "Ireland",
    "DR Congo": "Congo DR",
    "Cape Verde": "Cabo Verde",
    "Ivory Coast": "C\u00f4te d'Ivoire",
    "Cote d'Ivoire": "C\u00f4te d'Ivoire",
}
for c in ("home_team", "away_team"):
    RICH[c] = RICH[c].replace(NAME_MAP)
    NEW[c]  = NEW[c].replace(NAME_MAP)

print(f"Rich dataset: {len(RICH):,} matches, latest {RICH.date.max().date()}")
print(f"New  dataset: {len(NEW):,} matches, latest {NEW.date.max().date()}")
print(f"2026 World Cup matches in new dataset: {(NEW.tournament=='FIFA World Cup').sum() if 'tournament' in NEW.columns else '?'}")

Rich dataset: 23,921 matches, latest 2022-06-14
New  dataset: 49,477 matches, latest 2026-06-27
2026 World Cup matches in new dataset: 1036


## 4. Feature engineering on the rich dataset (training data)

In [5]:
df = RICH.sort_values("date").reset_index(drop=True).copy()

# Long-format per-team view for rolling form & goal differential
def expand(d):
    h = d[["date","home_team","home_team_score","away_team_score","home_team_result"]].rename(
        columns={"home_team":"team","home_team_score":"gf","away_team_score":"ga","home_team_result":"result"})
    a = d[["date","away_team","away_team_score","home_team_score","home_team_result"]].rename(
        columns={"away_team":"team","away_team_score":"gf","home_team_score":"ga"})
    a["result"] = a["home_team_result"].map({"Win":"Lose","Lose":"Win","Draw":"Draw"})
    a = a.drop(columns=["home_team_result"])
    return pd.concat([h,a], ignore_index=True).sort_values("date")

tv = expand(df)
tv["points"] = tv["result"].map({"Win":3,"Draw":1,"Lose":0})
tv["gd"]     = tv["gf"] - tv["ga"]
tv["form_5"] = tv.groupby("team")["points"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
tv["gd_5"]   = tv.groupby("team")["gd"].transform(    lambda s: s.shift().rolling(5, min_periods=1).mean())
form_lookup  = tv.set_index(["team","date"])[["form_5","gd_5"]]

def lookup_form(team, date):
    try:
        r = form_lookup.loc[(team, date)]
        if isinstance(r, pd.DataFrame): r = r.iloc[0]
        return float(r["form_5"] or 1.0), float(r["gd_5"] or 0.0)
    except KeyError:
        return 1.0, 0.0

df[["home_form_5","home_gd_5"]] = df.apply(lambda r: lookup_form(r.home_team, r.date), axis=1, result_type="expand")
df[["away_form_5","away_gd_5"]] = df.apply(lambda r: lookup_form(r.away_team, r.date), axis=1, result_type="expand")

In [6]:
# Running ELO over the historical rich dataset
elo = defaultdict(lambda: 1500.0); K = 24
he, ae = [], []
for _, r in df.iterrows():
    rh, ra = elo[r.home_team], elo[r.away_team]
    he.append(rh); ae.append(ra)
    eh = 1 / (1 + 10 ** ((ra - rh) / 400))
    sh = {"Win":1.0, "Draw":0.5, "Lose":0.0}[r.home_team_result]
    elo[r.home_team] = rh + K*(sh - eh)
    elo[r.away_team] = ra + K*((1-sh) - (1-eh))
df["home_elo"], df["away_elo"] = he, ae
print("ELO computed for the rich dataset.")

ELO computed for the rich dataset.


In [7]:
feat = [
    "home_team_fifa_rank", "away_team_fifa_rank",
    "home_team_total_fifa_points", "away_team_total_fifa_points",
    "home_team_goalkeeper_score", "away_team_goalkeeper_score",
    "home_team_mean_defense_score", "away_team_mean_defense_score",
    "home_team_mean_offense_score", "away_team_mean_offense_score",
    "home_team_mean_midfield_score", "away_team_mean_midfield_score",
    "home_form_5", "away_form_5", "home_gd_5", "away_gd_5",
    "home_elo", "away_elo",
]
data = df.dropna(subset=feat + ["home_team_result"]).copy()
data["rank_diff"]   = data.away_team_fifa_rank - data.home_team_fifa_rank
data["points_diff"] = data.home_team_total_fifa_points - data.away_team_total_fifa_points
data["elo_diff"]    = data.home_elo - data.away_elo
feat += ["rank_diff", "points_diff", "elo_diff"]

le = LabelEncoder().fit(["Lose","Draw","Win"])
X, y = data[feat], le.transform(data.home_team_result)
print(f"Training rows: {len(X):,}  |  features: {len(feat)}")

Training rows: 4,301  |  features: 21


## 5. Train XGBoost

In [8]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
model = xgb.XGBClassifier(
    objective="multi:softprob", num_class=3, n_estimators=600, max_depth=6,
    learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    eval_metric="mlogloss", tree_method="hist", random_state=RANDOM_STATE,
)
model.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
p, pr = model.predict(Xte), model.predict_proba(Xte)
print(f"Accuracy : {accuracy_score(yte, p):.3f}")
print(f"Precision: {precision_score(yte, p, average='weighted'):.3f}")
print(f"Log-loss : {log_loss(yte, pr):.3f}\n")
print(classification_report(yte, p, target_names=le.classes_))

Accuracy : 0.480
Precision: 0.449
Log-loss : 1.212

              precision    recall  f1-score   support

        Draw       0.27      0.16      0.20       218
        Lose       0.42      0.42      0.42       253
         Win       0.56      0.70      0.62       390

    accuracy                           0.48       861
   macro avg       0.42      0.43      0.42       861
weighted avg       0.45      0.48      0.46       861



## 6. Refresh ELO + rolling form with **post‑2022** matches

The new dataset only carries basic columns (date / teams / scores / tournament), so we can't recompute squad scores or FIFA ranks for the recent games — but we **can** update each team's running **ELO** and rolling **form / goal differential**, which are the most influential signals.

In [9]:
recent = NEW[(NEW.date > RICH.date.max()) & NEW.home_score.notna()].sort_values("date").copy()
print(f"Refreshing with {len(recent):,} matches from {recent.date.min().date()} \u2192 {recent.date.max().date()}")

# Extend the long-format view with new matches and recompute trailing form
def res_str(gf, ga): return "Win" if gf > ga else ("Lose" if gf < ga else "Draw")
extra = pd.DataFrame({
    "date": pd.concat([recent.date, recent.date]).values,
    "team": pd.concat([recent.home_team, recent.away_team]).values,
    "gf":   pd.concat([recent.home_score, recent.away_score]).values,
    "ga":   pd.concat([recent.away_score, recent.home_score]).values,
})
extra["gd"]     = extra.gf - extra.ga
extra["result"] = [res_str(g, a) for g, a in zip(extra.gf, extra.ga)]
extra["points"] = extra.result.map({"Win":3, "Draw":1, "Lose":0})
all_tv = pd.concat(
    [tv[["team","date","gf","ga","gd","result","points"]], extra], ignore_index=True
).sort_values(["team","date"]).reset_index(drop=True)
all_tv["form_5"] = all_tv.groupby("team")["points"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
all_tv["gd_5"]   = all_tv.groupby("team")["gd"].transform(    lambda s: s.shift().rolling(5, min_periods=1).mean())
latest_form = all_tv.sort_values("date").groupby("team").tail(1).set_index("team")[["form_5","gd_5"]]

# Continue ELO with the post-2022 matches
for _, r in recent.iterrows():
    rh, ra = elo[r.home_team], elo[r.away_team]
    eh = 1 / (1 + 10 ** ((ra - rh) / 400))
    sh = 1.0 if r.home_score > r.away_score else (0.0 if r.home_score < r.away_score else 0.5)
    elo[r.home_team] = rh + K*(sh - eh)
    elo[r.away_team] = ra + K*((1-sh) - (1-eh))

print("\nTop‑10 teams by **updated** ELO (as of today):")
print(pd.Series(dict(elo)).sort_values(ascending=False).head(10).round(0))

Refreshing with 4,008 matches from 2022-06-17 → 2026-06-22

Top‑10 teams by **updated** ELO (as of today):
Argentina    2000.0
Spain        1966.0
France       1934.0
Brazil       1906.0
England      1893.0
Germany      1886.0
Colombia     1883.0
Portugal     1881.0
Morocco      1874.0
Japan        1867.0
dtype: float64


## 7. Build a reusable `predict_match()`

Uses each team's latest known FIFA rank + squad scores (carried forward from the rich dataset) and the updated ELO + form.

In [10]:
H = df[["date","home_team","home_team_fifa_rank","home_team_total_fifa_points","home_team_goalkeeper_score",
         "home_team_mean_defense_score","home_team_mean_offense_score","home_team_mean_midfield_score"]].copy()
A = df[["date","away_team","away_team_fifa_rank","away_team_total_fifa_points","away_team_goalkeeper_score",
         "away_team_mean_defense_score","away_team_mean_offense_score","away_team_mean_midfield_score"]].copy()
H.columns = ["date","team","fifa_rank","fifa_points","gk","defc","off","mid"]
A.columns = ["date","team","fifa_rank","fifa_points","gk","defc","off","mid"]
latest = (
    pd.concat([H, A]).dropna(subset=["fifa_rank"]).sort_values("date")
    .groupby("team").tail(1).set_index("team")
)
print(f"Latest squad/rank info available for {len(latest)} teams.")

def predict_match(home: str, away: str):
    if home not in latest.index or away not in latest.index:
        missing = [t for t in (home, away) if t not in latest.index]
        return None, f"⚠️  no historical squad data for: {missing}"
    h, a = latest.loc[home], latest.loc[away]
    hf = latest_form.loc[home] if home in latest_form.index else pd.Series({"form_5":1.0, "gd_5":0.0})
    af = latest_form.loc[away] if away in latest_form.index else pd.Series({"form_5":1.0, "gd_5":0.0})
    he_, ae_ = elo.get(home, 1500.0), elo.get(away, 1500.0)
    row = {
        "home_team_fifa_rank": h.fifa_rank, "away_team_fifa_rank": a.fifa_rank,
        "home_team_total_fifa_points": h.fifa_points, "away_team_total_fifa_points": a.fifa_points,
        "home_team_goalkeeper_score": h.gk, "away_team_goalkeeper_score": a.gk,
        "home_team_mean_defense_score": h.defc, "away_team_mean_defense_score": a.defc,
        "home_team_mean_offense_score": h.off, "away_team_mean_offense_score": a.off,
        "home_team_mean_midfield_score": h.mid, "away_team_mean_midfield_score": a.mid,
        "home_form_5": hf.form_5, "away_form_5": af.form_5,
        "home_gd_5": hf.gd_5, "away_gd_5": af.gd_5,
        "home_elo": he_, "away_elo": ae_,
        "rank_diff": a.fifa_rank - h.fifa_rank,
        "points_diff": h.fifa_points - a.fifa_points,
        "elo_diff": he_ - ae_,
    }
    probs = model.predict_proba(pd.DataFrame([row])[feat])[0]
    return {cls: float(p) for cls, p in zip(le.classes_, probs)}, None

predict_match("Argentina", "France")

Latest squad/rank info available for 211 teams.


({np.str_('Draw'): 0.022032054141163826,
  np.str_('Lose'): 0.9107998013496399,
  np.str_('Win'): 0.06716816872358322},
 None)

## 8. 🏆 This week's 2026 FIFA World Cup fixtures — predictions

We pull every World Cup match scheduled within the next 7 days from `results_recent.csv`, generate XGBoost probabilities, and (if Foundry is enabled) ask the LLM for a short analyst take.

In [11]:
from datetime import datetime, timedelta
TODAY = pd.Timestamp(datetime.today().date())
WEEK_END = TODAY + pd.Timedelta(days=7)
print(f"Fixtures between {TODAY.date()} and {WEEK_END.date()}\n" + "-"*70)

fixtures = NEW[(NEW.tournament == "FIFA World Cup")
               & (NEW.date >= TODAY) & (NEW.date <= WEEK_END)
               & NEW.home_score.isna()].sort_values(["date","home_team"]).copy()

rows = []
for _, r in fixtures.iterrows():
    probs, err = predict_match(r.home_team, r.away_team)
    if err:
        print(f"  {r.date.date()}  {r.home_team:25s} vs {r.away_team:25s}  {err}")
        continue
    winner = r.home_team if probs["Win"] >= probs["Lose"] else r.away_team
    if max(probs.values()) == probs["Draw"]:
        winner = "Draw"
    print(f"  {r.date.date()}  {r.home_team:25s} vs {r.away_team:25s}  "
          f"W{probs['Win']:.2f} D{probs['Draw']:.2f} L{probs['Lose']:.2f}  \u2192 {winner}")
    rows.append({"date": r.date.date(), "home": r.home_team, "away": r.away_team,
                  "P_home": probs["Win"], "P_draw": probs["Draw"], "P_away": probs["Lose"],
                  "prediction": winner, "city": r.city})
predictions = pd.DataFrame(rows)
predictions

Fixtures between 2026-06-23 and 2026-06-30
----------------------------------------------------------------------


  2026-06-23  Colombia                  vs Congo DR                   W0.92 D0.04 L0.04  → Colombia
  2026-06-23  England                   vs Ghana                      W0.58 D0.39 L0.03  → England
  2026-06-23  Panama                    vs Croatia                    W0.14 D0.08 L0.77  → Croatia
  2026-06-23  Portugal                  vs Uzbekistan                 W0.62 D0.12 L0.26  → Portugal
  2026-06-24  Bosnia and Herzegovina    vs Qatar                      W0.17 D0.22 L0.61  → Qatar
  2026-06-24  Canada                    vs Switzerland                W0.29 D0.31 L0.40  → Switzerland
  2026-06-24  Mexico                    vs Czech Republic             W0.39 D0.44 L0.17  → Draw


  2026-06-24  Morocco                   vs Haiti                      W0.92 D0.04 L0.04  → Morocco
  2026-06-24  Scotland                  vs Brazil                     W0.19 D0.33 L0.48  → Brazil
  2026-06-24  South Africa              vs South Korea                W0.77 D0.15 L0.08  → South Africa
  2026-06-25  Curaçao                   vs Côte d'Ivoire              W0.79 D0.14 L0.07  → Curaçao
  2026-06-25  Ecuador                   vs Germany                    W0.03 D0.34 L0.63  → Germany
  2026-06-25  Japan                     vs Sweden                     W0.60 D0.11 L0.29  → Japan
  2026-06-25  Paraguay                  vs Australia                  W0.85 D0.07 L0.07  → Paraguay
  2026-06-25  Tunisia                   vs Netherlands                W0.05 D0.22 L0.73  → Netherlands
  2026-06-25  USA                       vs Turkey                     W0.88 D0.09 L0.04  → USA
  2026-06-26  Cabo Verde                vs Saudi Arabia               W0.67 D0.28 L0.06  → Cabo Verde
  20

,date,home,away,P_home,P_draw,P_away,prediction,city
0,2026-06-23,Colombia,Congo DR,0.915453,0.044552,0.039994,Colombia,Zapopan
1,2026-06-23,England,Ghana,0.577211,0.388681,0.034107,England,Foxborough
2,2026-06-23,Panama,Croatia,0.144470,0.083559,0.771971,Croatia,Toronto
3,2026-06-23,Portugal,Uzbekistan,0.618312,0.116964,0.264724,Portugal,Houston
4,2026-06-24,Bosnia and Herzegovina,Qatar,0.169623,0.217831,0.612547,Qatar,Seattle
5,2026-06-24,Canada,Switzerland,0.291279,0.305747,0.402974,Switzerland,Vancouver
6,2026-06-24,Mexico,Czech Republic,0.387662,0.438161,0.174177,Draw,Mexico City
7,2026-06-24,Morocco,Haiti,0.922431,0.038243,0.039327,Morocco,Atlanta
8,2026-06-24,Scotland,Brazil,0.192905,0.329129,0.477966,Brazil,Miami Gardens
9,2026-06-24,South Africa,South Korea,0.770324,0.149259,0.080417,South Africa,Guadalupe


In [12]:
# Highest-confidence picks of the week
if not predictions.empty:
    predictions["top_prob"] = predictions[["P_home","P_draw","P_away"]].max(axis=1)
    print("\nTop-5 most confident picks this week:")
    print(predictions.sort_values("top_prob", ascending=False).head(5)[["date","home","away","prediction","top_prob"]].to_string(index=False))
    print("\nTightest matches this week:")
    print(predictions.sort_values("top_prob").head(5)[["date","home","away","prediction","top_prob"]].to_string(index=False))


Top-5 most confident picks this week:
      date     home      away prediction  top_prob
2026-06-24  Morocco     Haiti    Morocco  0.922431
2026-06-23 Colombia  Congo DR   Colombia  0.915453
2026-06-27  Croatia     Ghana    Croatia  0.908339
2026-06-25      USA    Turkey        USA  0.875532
2026-06-25 Paraguay Australia   Paraguay  0.854680

Tightest matches this week:
      date     home           away  prediction  top_prob
2026-06-26  Uruguay          Spain       Spain  0.356359
2026-06-24   Canada    Switzerland Switzerland  0.402974
2026-06-24   Mexico Czech Republic        Draw  0.438161
2026-06-26    Egypt           Iran       Egypt  0.465700
2026-06-24 Scotland         Brazil      Brazil  0.477966


## 9. Optional — ask the Foundry LLM for analyst commentary

In [13]:
SYSTEM = ("You are an expert football pundit. Given two national teams, their stats, and "
          "a model's win/draw/lose probabilities, write 2-3 punchy sentences naming the "
          "likely winner and the key reason.")

if foundry_client is not None and not predictions.empty:
    pick = predictions.iloc[0]
    prompt = (f"Match: {pick.home} vs {pick.away} on {pick.date}.\n"
              f"Probabilities: home win {pick.P_home:.2f}, draw {pick.P_draw:.2f}, away win {pick.P_away:.2f}.\n"
              f"My model picks {pick.prediction}. Give a short analyst take.")
    print(ask_foundry(SYSTEM, prompt) or "(Foundry returned no content)")
else:
    print("Foundry not configured \u2014 skipping LLM commentary. Set AZURE_AI_ENDPOINT + AZURE_AI_API_KEY to enable.")

Foundry not configured — skipping LLM commentary. Set AZURE_AI_ENDPOINT + AZURE_AI_API_KEY to enable.


---
### Next steps / ideas

* Pull `results_recent.csv` automatically from `https://raw.githubusercontent.com/martj42/international_results/master/results.csv` each run for always-fresh data.
* Swap `FOUNDRY_MODEL` for `o3-mini` or `Phi-4` and compare commentary.
* Run knockout fixtures as a 1000-trial Monte-Carlo simulation instead of greedy argmax.
* Train a parallel LightGBM or shallow neural net and ensemble.
* Add player-level injury / form data once available.